In [12]:
"""
Multi-Year Predictive Analytics of Employee Earnings in Boston (2022-2024)
Authors: Dev Panchal, Om Patel
Course: ALY 6140 - Python & Analytics
Northeastern University, Fall 2025

INSTRUCTIONS:
1. Ensure capstone_group_dev_om.py is in the same directory
2. Download data files or ensure URLs are accessible
3. Run: python capstone_group_dev_om_analysis.py
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.arima.model import ARIMA
import warnings
import os
warnings.filterwarnings('ignore')

from capstone_group_dev_om import (
    load_and_combine_datasets,
    load_and_combine_local_datasets,
    preprocess_employee_data,
    evaluate_and_compare_models,
    generate_data_quality_report,
    create_comprehensive_eda_plots,
    perform_statistical_tests
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

OUTPUT_DIR = 'analysis_outputs'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def main():
    print("="*80)
    print("CAPSTONE PROJECT: EMPLOYEE EARNINGS ANALYSIS")
    print("Authors: Dev Panchal, Om Patel")
    print("="*80)
    
    # Step 1: Data Extraction
    print("\n" + "="*80)
    print("STEP 1: DATA EXTRACTION")
    print("="*80 + "\n")
    
    data_urls = {
        2024: 'https://data.boston.gov/dataset/418983dc-7cae-42bb-88e4-d56f5adcf869/resource/579a4be3-9ca7-4183-bc95-7d67ee715b6d/download/employee_earnings_report_2024.csv',
        2023: 'https://data.boston.gov/dataset/418983dc-7cae-42bb-88e4-d56f5adcf869/resource/6b3c5333-1dcb-4b3d-9cd7-6a03fb526da7/download/employee-earnings-report-2023.xlsx',
        2022: 'https://data.boston.gov/dataset/418983dc-7cae-42bb-88e4-d56f5adcf869/resource/63ac638b-36c4-487d-9453-1d83eb5090d2/download/finalconsolidatedcy22earnings_feb2023.xlsx-sheet1.csv'
    }
    
    data_files = {
        2024: 'employee_earnings_2024.csv',
        2023: 'employee_earnings_2023.csv',
        2022: 'employee_earnings_2022.csv'
    }
    
    try:
        df_raw = load_and_combine_datasets(data_urls, year_column='Year')
    except:
        df_raw = load_and_combine_local_datasets(data_files, year_column='Year')
    
    print("\n📊 Sample Data:")
    print(df_raw.head())
    
    # Step 2: Data Quality
    print("\n" + "="*80)
    print("STEP 2: DATA QUALITY ASSESSMENT")
    print("="*80 + "\n")
    
    quality_report = generate_data_quality_report(df_raw)
    
    # Step 3: Data Cleaning
    print("\n" + "="*80)
    print("STEP 3: DATA CLEANING")
    print("="*80 + "\n")
    
    config = {
        'fill_missing_with_zero': ['OVERTIME', 'RETRO', 'OTHER', 'DETAIL', 'QUINN_EDUCATION', 'INJURED'],
        'create_features': True,
        'rename_columns': True,
        'drop_duplicates': True
    }
    
    df_clean, stats = preprocess_employee_data(df_raw, config=config)
    
    print("\n✨ Cleaned Data:")
    print(df_clean.head())
    
    print("\n📈 Summary Statistics:")
    key_cols = ['REGULAR', 'OVERTIME', 'TOTAL GROSS', 'OVERTIME_PCT']
    available = [c for c in key_cols if c in df_clean.columns]
    print(df_clean[available].describe())
    
    # Step 4: EDA
    print("\n" + "="*80)
    print("STEP 4: EXPLORATORY DATA ANALYSIS")
    print("="*80 + "\n")
    
    eda_figures = create_comprehensive_eda_plots(df_clean, target_col='TOTAL GROSS')
    
    print("\n💾 Saving visualizations...")
    for name, fig in eda_figures.items():
        filename = f"{OUTPUT_DIR}/eda_{name}.png"
        fig.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"   ✅ {filename}")
        plt.close(fig)
    
    print("\n📊 Year-over-Year:")
    yearly = df_clean.groupby('YEAR').agg({
        'TOTAL GROSS': ['mean', 'median', 'sum', 'count'],
        'OVERTIME': 'mean'
    }).round(2)
    print(yearly)
    
    # Step 5: Statistical Tests
    print("\n" + "="*80)
    print("STEP 5: STATISTICAL TESTS")
    print("="*80 + "\n")
    
    stat_results = perform_statistical_tests(df_clean, 'DEPARTMENT_NAME', 'TOTAL GROSS', 10)
    
    # Step 6: Modeling Prep
    print("\n" + "="*80)
    print("STEP 6: MODELING PREPARATION")
    print("="*80 + "\n")
    
    print("🔧 Using YEAR + DEPARTMENT + TITLE (avoiding multicollinearity)")
    
    X = df_clean[['YEAR']].copy()
    y = df_clean['TOTAL GROSS'].copy()
    
    if y.dtype == 'object':
        y = pd.to_numeric(y.astype(str).str.replace(',', ''), errors='coerce')
    
    X = X.fillna(0)
    y = y.fillna(y.median())
    
    if 'DEPARTMENT_NAME' in df_clean.columns:
        top_depts = df_clean['DEPARTMENT_NAME'].value_counts().head(15).index
        df_clean['DEPT_CAT'] = df_clean['DEPARTMENT_NAME'].apply(
            lambda x: x if x in top_depts else 'Other'
        )
        dept_dummies = pd.get_dummies(df_clean['DEPT_CAT'], prefix='DEPT', drop_first=True)
        X = pd.concat([X, dept_dummies], axis=1)
        print(f"   ✅ Added {len(dept_dummies.columns)} departments")
    
    if 'TITLE' in df_clean.columns:
        top_titles = df_clean['TITLE'].value_counts().head(10).index
        df_clean['TITLE_CAT'] = df_clean['TITLE'].apply(
            lambda x: x if x in top_titles else 'Other'
        )
        title_dummies = pd.get_dummies(df_clean['TITLE_CAT'], prefix='TITLE', drop_first=True)
        X = pd.concat([X, title_dummies], axis=1)
        print(f"   ✅ Added {len(title_dummies.columns)} titles")
    
    print(f"\n📊 Features: {X.shape[1]}, Samples: {X.shape[0]:,}")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    print(f"   Train: {X_train.shape[0]:,}, Test: {X_test.shape[0]:,}")
    
    # Step 7: Model Training
    print("\n" + "="*80)
    print("STEP 7: MODEL TRAINING")
    print("="*80 + "\n")
    
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
    }
    
    results_df, trained_models, predictions = evaluate_and_compare_models(
        models, X_train, X_test, y_train, y_test
    )
    
    if len(results_df) == 0:
        raise ValueError("No models trained successfully")
    
    print("\n📊 Results:")
    print(results_df.to_string(index=False))
    
    # Step 8: LR Coefficients
    print("\n" + "="*80)
    print("STEP 8: LINEAR REGRESSION COEFFICIENTS")
    print("="*80 + "\n")
    
    lr_model = trained_models['Linear Regression']
    coefs = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': lr_model.coef_
    }).sort_values('Coefficient', key=abs, ascending=False)
    
    print("Top 10 Coefficients:")
    print(coefs.head(10).to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(12, 6))
    top10 = coefs.head(10)
    colors = ['green' if x > 0 else 'red' for x in top10['Coefficient']]
    ax.barh(range(len(top10)), top10['Coefficient'], color=colors, alpha=0.7)
    ax.set_yticks(range(len(top10)))
    ax.set_yticklabels(top10['Feature'])
    ax.set_xlabel('Coefficient Value', fontweight='bold')
    ax.set_title('Linear Regression Coefficients', fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='--')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/lr_coefficients.png', dpi=300, bbox_inches='tight')
    print(f"   ✅ Saved: {OUTPUT_DIR}/lr_coefficients.png")
    plt.close()
    
    # Step 9: RF Importance
    print("\n" + "="*80)
    print("STEP 9: RANDOM FOREST IMPORTANCE")
    print("="*80 + "\n")
    
    rf_model = trained_models['Random Forest']
    importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("Top 10 Features:")
    print(importance.head(10).to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(12, 6))
    top10 = importance.head(10)
    ax.barh(range(len(top10)), top10['Importance'], color='forestgreen', alpha=0.7)
    ax.set_yticks(range(len(top10)))
    ax.set_yticklabels(top10['Feature'])
    ax.set_xlabel('Importance', fontweight='bold')
    ax.set_title('Random Forest Feature Importance', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/rf_feature_importance.png', dpi=300, bbox_inches='tight')
    print(f"   ✅ Saved: {OUTPUT_DIR}/rf_feature_importance.png")
    plt.close()
    
    # Step 10: Predictions Plot
    print("\n" + "="*80)
    print("STEP 10: PREDICTION VISUALIZATION")
    print("="*80 + "\n")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, (name, model) in enumerate(trained_models.items()):
        y_pred = predictions[name]['test']
        r2 = results_df[results_df['Model'] == name]['Test_R2'].values[0]
        
        axes[idx].scatter(y_test, y_pred, alpha=0.3, s=10)
        axes[idx].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
                       'r--', lw=2, label='Perfect')
        axes[idx].set_xlabel('Actual ($)', fontweight='bold')
        axes[idx].set_ylabel('Predicted ($)', fontweight='bold')
        axes[idx].set_title(f'{name} (R²={r2:.4f})', fontweight='bold')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/actual_vs_predicted.png', dpi=300, bbox_inches='tight')
    print(f"   ✅ Saved: {OUTPUT_DIR}/actual_vs_predicted.png")
    plt.close()
    
    # Step 11: ARIMA
    print("\n" + "="*80)
    print("STEP 11: ARIMA FORECASTING")
    print("="*80 + "\n")
    
    ts_data = df_clean.groupby('YEAR')['TOTAL GROSS'].sum().reset_index()
    ts_data.columns = ['Year', 'Total_Earnings']
    
    print("Annual Earnings:")
    print(ts_data.to_string(index=False))
    
    forecast = None
    yoy_growth = None
    
    print("\n🔍 Filtering valid data...")
    valid = ts_data[ts_data['Total_Earnings'] > 1000000]
    
    if len(valid) < len(ts_data):
        print(f"   Excluded {len(ts_data) - len(valid)} year(s) with incomplete data")
        ts_data = valid
    
    if len(ts_data) >= 2:
        ts = ts_data.set_index('Year')['Total_Earnings']
        
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(ts.index, ts.values, marker='o', linewidth=2, markersize=10, color='darkblue')
        ax.set_xlabel('Year', fontweight='bold')
        ax.set_ylabel('Total Earnings ($)', fontweight='bold')
        ax.set_title('Historical Earnings', fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e9:.2f}B'))
        for x, y in zip(ts.index, ts.values):
            ax.text(x, y, f'${y/1e9:.2f}B', ha='center', va='bottom', fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/time_series_historical.png', dpi=300, bbox_inches='tight')
        print(f"\n   ✅ Saved: {OUTPUT_DIR}/time_series_historical.png")
        plt.close()
        
        print(f"\n📊 Fitting ARIMA with {len(ts)} years...")
        
        try:
            arima_model = ARIMA(ts, order=(1, 1, 0))
            fitted = arima_model.fit()
            
            print("\nARIMA Summary:")
            print(fitted.summary())
            
            forecast = fitted.forecast(steps=2)
            forecast_years = np.arange(ts.index[-1] + 1, ts.index[-1] + 3)
            
            print(f"\n🔮 Forecast:")
            for yr, val in zip(forecast_years, forecast):
                print(f"   {yr}: ${val/1e9:.2f}B")
            
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.plot(ts.index, ts.values, marker='o', linewidth=2, color='darkblue', label='Historical')
            ax.plot(forecast_years, forecast.values, marker='s', linewidth=2, color='red', linestyle='--', label='Forecast')
            ax.set_xlabel('Year', fontweight='bold')
            ax.set_ylabel('Total Earnings ($)', fontweight='bold')
            ax.set_title('ARIMA Forecast', fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.legend()
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e9:.2f}B'))
            plt.tight_layout()
            plt.savefig(f'{OUTPUT_DIR}/arima_forecast.png', dpi=300, bbox_inches='tight')
            print(f"\n   ✅ Saved: {OUTPUT_DIR}/arima_forecast.png")
            plt.close()
            
            if ts.values[0] != 0:
                yoy_growth = ((forecast.iloc[0] - ts.values[-1]) / ts.values[-1]) * 100
                print(f"\n📈 Projected Growth: {yoy_growth:.2f}%")
            
        except Exception as e:
            print(f"\n❌ ARIMA Error: {str(e)}")
    else:
        print("\n⚠️  Insufficient data for ARIMA")
    
    # Step 12: Key Findings
    print("\n" + "="*80)
    print("STEP 12: KEY FINDINGS")
    print("="*80 + "\n")
    
    avg_sal = df_clean['TOTAL GROSS'].mean()
    med_sal = df_clean['TOTAL GROSS'].median()
    ot_earners = (df_clean['OVERTIME'] > 0).sum()
    ot_pct = (ot_earners / len(df_clean)) * 100
    avg_ot = df_clean[df_clean['OVERTIME'] > 0]['OVERTIME'].mean()
    
    if 'DEPARTMENT_NAME' in df_clean.columns:
        top_dept = df_clean.groupby('DEPARTMENT_NAME')['TOTAL GROSS'].mean().idxmax()
        top_pay = df_clean.groupby('DEPARTMENT_NAME')['TOTAL GROSS'].mean().max()
    
    best_model = results_df.iloc[0]['Model']
    best_r2 = results_df.iloc[0]['Test_R2']
    best_rmse = results_df.iloc[0]['Test_RMSE']
    
    print("💰 COMPENSATION:")
    print(f"   Average: ${avg_sal:,.2f}")
    print(f"   Median: ${med_sal:,.2f}")
    print(f"   Overtime Recipients: {ot_earners:,} ({ot_pct:.1f}%)")
    print(f"   Avg Overtime: ${avg_ot:,.2f}")
    
    print("\n🏢 DEPARTMENTS:")
    print(f"   Highest Paying: {top_dept}")
    print(f"   Average: ${top_pay:,.2f}")
    
    print("\n🤖 MODELS:")
    print(f"   Best: {best_model}")
    print(f"   R²: {best_r2:.4f} ({best_r2*100:.2f}%)")
    print(f"   RMSE: ${best_rmse:,.2f}")
    
    print("\n📊 SIGNIFICANCE:")
    if stat_results.get('anova'):
        p = stat_results['anova']['p_value']
        print(f"   ANOVA p-value: {p:.6f}")
        print(f"   Result: {'SIGNIFICANT' if p < 0.05 else 'NOT SIGNIFICANT'}")
    
    print("\n🔮 FORECAST:")
    if forecast is not None:
        print(f"   2025 Projection: ${forecast.iloc[0]/1e9:.2f}B")
        if yoy_growth:
            print(f"   Growth: {yoy_growth:.2f}%")
    
    # Step 13: Save CSVs
    print("\n" + "="*80)
    print("STEP 13: SAVING RESULTS")
    print("="*80 + "\n")
    
    results_df.to_csv(f'{OUTPUT_DIR}/model_comparison.csv', index=False)
    importance.to_csv(f'{OUTPUT_DIR}/feature_importance.csv', index=False)
    coefs.to_csv(f'{OUTPUT_DIR}/lr_coefficients.csv', index=False)
    yearly.to_csv(f'{OUTPUT_DIR}/yearly_summary.csv')
    
    print("   ✅ model_comparison.csv")
    print("   ✅ feature_importance.csv")
    print("   ✅ lr_coefficients.csv")
    print("   ✅ yearly_summary.csv")
    
    print("\n" + "="*80)
    print("🎉 ANALYSIS COMPLETE!")
    print("="*80)
    
    return {
        'df_clean': df_clean,
        'results_df': results_df,
        'models': trained_models,
        'importance': importance,
        'coefficients': coefs
    }

if __name__ == "__main__":
    try:
        results = main()
        print("\n✅ Success! Check analysis_outputs/ folder")
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "="*80)
    print("END OF ANALYSIS")
    print("="*80)

CAPSTONE PROJECT: EMPLOYEE EARNINGS ANALYSIS
Authors: Dev Panchal, Om Patel

STEP 1: DATA EXTRACTION

LOADING AND COMBINING DATASETS FROM URLS

Loading data for year 2024...
   Loaded 25,530 records with 13 columns

Loading data for year 2023...
   Loaded 25,812 records with 13 columns

Loading data for year 2022...
   Loaded 23,204 records with 13 columns

SUCCESSFULLY COMBINED 3 DATASETS
   Total Records: 74,546
   Year Range: 2022 - 2024


📊 Sample Data:
               NAME           DEPARTMENT_NAME                        TITLE  \
0  Demesmin,Stanley  Boston Police Department      Police Lieutenant (Det)   
1   Sordillo,Paul J     Facilities Management  Building Services Fleet Mgr   
2      Smith,Sean P  Boston Police Department            Police Lieutenant   
3  Connolly,Timothy  Boston Police Department           Police Captain/DDC   
4  Danilecki,John H  Boston Police Department               Police Captain   

      REGULAR       RETRO       OTHER    OVERTIME INJURED     DETAIL 